# Cálculo SIMAT: días limpios del índice de calidad del aire, 2019–2025

Réplica del conteo anual de días limpios que publica el SIMAT en sus Informes Anuales de Calidad del Aire, calculado a partir de las concentraciones horarias por estación y aplicando en cada fecha la norma vigente.


In [ ]:
# Código de cálculo. Las fórmulas y normas que aplica se describen en la sección 3.
from pathlib import Path

import numpy as np
import pandas as pd

# ---------------------------------------------------------------- datos
ARCHIVOS = {  # concentraciones horarias por estación del SIMAT, en la carpeta datos/
    2019: Path("datos/contaminantes_2019.csv.gz"),
    2020: Path("datos/contaminantes_2020.csv.gz"),
    2021: Path("datos/contaminantes_2021.csv.gz"),
    2022: Path("datos/contaminantes_2022.zip"),
    2023: Path("datos/contaminantes_2023.zip"),
    2024: Path("datos/contaminantes_2024.zip"),
    2025: Path("datos/contaminantes_2025.zip"),
}
TRANSPORTE = {"ACO", "INN", "MPA", "AJU", "MON"}  # estaciones de transporte: no reportan el índice

# ---------------------------------------------------------------- normas
# (inicio de vigencia, norma) para la construcción del índice diario
NORMAS = [
    ("2019-01-01", "NADF-009-AIRE-2017"),
    ("2020-01-01", "NOM-172-SEMARNAT-2019"),
    ("2024-07-23", "NOM-172-SEMARNAT-2023"),
]
CATEGORIAS = {0: "Buena", 1: "Aceptable", 2: "Mala", 3: "Muy Mala", 4: "Ext. Mala"}

# Límites superiores de Buena, Aceptable, Mala y Muy Mala
LIM_2019 = {  # NOM-172-SEMARNAT-2019, tablas 4 a 9 (informe 2021, tabla 5.2)
    "O3_1h": [0.051, 0.095, 0.135, 0.175], "O3_8h": [0.051, 0.070, 0.092, 0.114],
    "NO2_1h": [0.107, 0.210, 0.230, 0.250], "SO2_24h": [0.008, 0.110, 0.165, 0.220],
    "CO_8h": [8.75, 11.00, 13.30, 15.50], "PM10": [50, 75, 155, 235], "PM25": [25, 45, 79, 147],
}
LIM_2023 = {  # NOM-172-SEMARNAT-2023, tablas 4 a 9
    "O3_1h": [0.058, 0.090, 0.135, 0.175], "NO2_1h": [0.053, 0.106, 0.160, 0.213],
    "SO2_1h": [0.035, 0.075, 0.185, 0.304], "CO_8h": [5.00, 9.00, 12.00, 16.00],
    "PM10": [45, 60, 132, 213], "PM25": [15, 33, 79, 130],             # a partir de enero de 2024
    "PM10_2026": [45, 50, 132, 213], "PM25_2026": [15, 25, 79, 130],   # a partir de enero de 2026
}
# NADF-009-AIRE-2017: concentración que corresponde a 100, 150 y 200 puntos del índice (informe 2019, págs. 76, 95 y 101)
LIM_NADF = {"O3_1h": [95, 154, 204], "PM10": [75, 214, 354], "PM25": [45, 97.4, 150.4]}  # ppb y µg/m³

# Cifras publicadas en los Informes Anuales de Calidad del Aire (páginas del PDF)
INFORME = pd.DataFrame({
    #        limpios, malos, Muy Mala o peor, Mala, Muy Mala, Ext. Mala, O3, PM10, PM2.5
    2019: [94, 271, 12, None, None, None, 204, 175, 55],   # págs. 76 y 78
    2020: [125, 241, 12, 229, 12, 0, 208, 68, 10],         # págs. 102-103
    2021: [135, 230, 55, 175, 49, 6, 204, 101, 13],        # págs. 108-109
    2022: [126, 239, 72, 167, 60, 12, 222, 89, 10],        # págs. 120-123
    2023: [120, 245, 65, 180, 61, 4, 228, 82, 10],         # págs. 110-112
}, index=["Limpios", "Malos", "Muy Mala o peor", "Mala", "Muy Mala", "Ext. Mala", "O3", "PM10", "PM2.5"]).T.astype("Int64")


# ---------------------------------------------------------------- lectura
def leer(ruta):
    """Concentraciones horarias en formato ancho: índice = fin de la hora, columnas = (contaminante, estación)."""
    d = pd.read_csv(ruta, header=None, names=range(5), dtype=str, encoding="latin1")
    fila = d.index[d[0] == "date"][0]  
    d = d.iloc[fila + 1:].set_axis(["date", "id_station", "id_parameter", "value", "unit"], axis=1)
    d = d.reset_index(drop=True)
    fecha = d["date"]
    if "/" in fecha.dropna().iloc[0]:  
        ts = pd.to_datetime(fecha.str[:10], format="%d/%m/%Y") + pd.to_timedelta(fecha.str[11:13].astype(float), unit="h")
    else:
        ts = pd.to_datetime(fecha, format="ISO8601")
    vacia = ts.isna()  
    ts = ts.ffill() + pd.to_timedelta(vacia.astype(int), unit="h")
    if ts.min().hour == 0:  
        ts = ts + pd.Timedelta(hours=1)
    d = d.assign(ts=ts, value=pd.to_numeric(d["value"], errors="coerce"))
    d = d[~d["id_station"].isin(TRANSPORTE)]
    ancho = d.pivot_table(index="ts", columns=["id_parameter", "id_station"], values="value", aggfunc="mean")
    return ancho.reindex(pd.date_range(ancho.index.min(), ancho.index.max(), freq="h"))


def redondear(x, decimales):
    """NOM-172 (5.2.4): si la cifra siguiente es 5 o más, se incrementa (104.5 -> 105)."""
    f = 10 ** decimales
    return np.floor(x * f + 0.5 + 1e-9) / f


def categoria(x, limites):
    """Número de límites superados: 0 = Buena ... 4 = Ext. Mala; NaN si no hay dato."""
    c = np.digitize(x, limites, right=True).astype(float)
    c[np.isnan(x)] = np.nan
    return pd.DataFrame(c, index=x.index, columns=x.columns)


def dia_de(indice):
    """Día al que pertenece cada hora (la hora 24 es 00:00 del día siguiente)."""
    return (indice - pd.Timedelta(hours=1)).normalize()


def movil(x, n):
    """Promedio móvil de n horas con al menos 75% de horas válidas (6 de 8, 18 de 24)."""
    return x.rolling(n, min_periods=int(0.75 * n)).mean()


def promedio_dia(x):
    """Promedio de 24 h de las horas 1 a 24 de cada día, con al menos 18 horas válidas. Índice = día."""
    g = x.groupby(dia_de(x.index))
    return g.mean().where(g.count() >= 18)


def max_dia(c, anio):
    """Categoría máxima del día entre estaciones y horas."""
    s = c.max(axis=1)
    if s.index.hour.nunique() > 1:  
        s = s.groupby(dia_de(s.index)).max()
    return s[s.index.year == anio]


def nadf2017(w, anio):
    """Índice de Calidad del Aire NADF-009-AIRE-2017: O3 horario, PM10 y PM2.5 con el promedio móvil de 24 h.
    1 = Buena o Regular (hasta 100 puntos), 2 = Mala, 3 = Muy Mala, 4 = Ext. Mala o peor."""
    d = pd.DataFrame({
        "O3_1h": max_dia(categoria(redondear(w["O3"], 0), LIM_NADF["O3_1h"]), anio),
        "PM10": max_dia(categoria(redondear(movil(w["PM10"], 24), 0), LIM_NADF["PM10"]), anio),
        "PM25": max_dia(categoria(redondear(movil(w["PM2.5"], 24), 1), LIM_NADF["PM25"]), anio),
    }) + 1
    d["O3"] = d["O3_1h"]
    return d


def nom2019(w, anio):
    """Índice AIRE Y SALUD, NOM-172-SEMARNAT-2019; partículas con el promedio de 24 h (NOM-025-SSA1-2014)."""
    o3, no2, so2 = w["O3"] / 1000, w["NO2"] / 1000, w["SO2"] / 1000  # ppb a ppm
    d = pd.DataFrame({
        "O3_1h": max_dia(categoria(redondear(o3, 3), LIM_2019["O3_1h"]), anio),
        "O3_8h": max_dia(categoria(redondear(movil(o3, 8), 3), LIM_2019["O3_8h"]), anio),
        "NO2_1h": max_dia(categoria(redondear(no2, 3), LIM_2019["NO2_1h"]), anio),
        "SO2_24h": max_dia(categoria(redondear(movil(so2, 24), 3), LIM_2019["SO2_24h"]), anio),
        "CO_8h": max_dia(categoria(redondear(movil(w["CO"], 8), 2), LIM_2019["CO_8h"]), anio),
        "PM10": max_dia(categoria(redondear(promedio_dia(w["PM10"]), 0), LIM_2019["PM10"]), anio),
        "PM25": max_dia(categoria(redondear(promedio_dia(w["PM2.5"]), 0), LIM_2019["PM25"]), anio),
    })
    d["O3"] = d[["O3_1h", "O3_8h"]].max(axis=1)  
    return d


def nom2023(w, anio):
    """Índice AIRE Y SALUD, NOM-172-SEMARNAT-2023, reporte diario (tabla 3): máximo horario de O3, NO2 y SO2,
    máximo del promedio móvil de 8 h de CO y promedio de 24 h de partículas."""
    o3, no2, so2 = w["O3"] / 1000, w["NO2"] / 1000, w["SO2"] / 1000
    d = {
        "O3_1h": max_dia(categoria(redondear(o3, 3), LIM_2023["O3_1h"]), anio),
        "NO2_1h": max_dia(categoria(redondear(no2, 3), LIM_2023["NO2_1h"]), anio),
        "SO2_1h": max_dia(categoria(redondear(so2, 3), LIM_2023["SO2_1h"]), anio),
        "CO_8h": max_dia(categoria(redondear(movil(w["CO"], 8), 2), LIM_2023["CO_8h"]), anio),
    }
    for clave, param in (("PM10", "PM10"), ("PM25", "PM2.5")):
        prom = redondear(promedio_dia(w[param]), 0)
        desde_2026 = np.broadcast_to(np.asarray(prom.index >= "2026-01-01")[:, None], prom.shape)
        c = categoria(prom, LIM_2023[clave]).mask(desde_2026, categoria(prom, LIM_2023[clave + "_2026"]))
        d[clave] = max_dia(c, anio)
    d = pd.DataFrame(d)
    d["O3"] = d["O3_1h"]
    return d


CALCULOS = {"NADF-009-AIRE-2017": nadf2017, "NOM-172-SEMARNAT-2019": nom2019, "NOM-172-SEMARNAT-2023": nom2023}


def norma_vigente(dias):
    inicio = pd.to_datetime([f for f, _ in NORMAS])
    return pd.Series([NORMAS[i][1] for i in inicio.searchsorted(dias, side="right") - 1], index=dias)


def indice_diario(w, anio, norma=None):
    """Categoría de cada día con la norma vigente en esa fecha (o con la norma indicada para todo el año)."""
    dias = pd.date_range(f"{anio}-01-01", f"{anio}-12-31")
    vigente = norma_vigente(dias) if norma is None else pd.Series(norma, index=dias)
    partes = [CALCULOS[n](w, anio).reindex(dias)[vigente == n].assign(Norma=n) for n in vigente.unique()]
    d = pd.concat(partes).sort_index()
    d["Total"] = d.drop(columns=["Norma", "O3"]).max(axis=1)
    return d


def resumen(diario):
    filas = {}
    for anio, d in diario.items():
        t = d["Total"]
        filas[anio] = {
            "Limpios": (t <= 1).sum(), "Malos": (t >= 2).sum(), "Muy Mala o peor": (t >= 3).sum(),
            "Mala": (t == 2).sum(), "Muy Mala": (t == 3).sum(), "Ext. Mala": (t == 4).sum(),
            "O3": (d["O3"] >= 2).sum(), "PM10": (d["PM10"] >= 2).sum(), "PM2.5": (d["PM25"] >= 2).sum(),
        }
    return pd.DataFrame(filas).T.astype("Int64")


datos = {anio: leer(ruta) for anio, ruta in ARCHIVOS.items()}
diario = {anio: indice_diario(w, anio) for anio, w in datos.items()}                           
calculado = resumen(diario)
norma_anio = {anio: " + ".join(d["Norma"].unique()) for anio, d in diario.items()}

#### 1.1 Días limpios

In [2]:
anios = INFORME.index
pd.DataFrame({
    "Norma": pd.Series(norma_anio)[anios],
    "Calculado": calculado.loc[anios, "Limpios"],
    "Informe": INFORME["Limpios"],
    "Diferencia": calculado.loc[anios, "Limpios"] - INFORME["Limpios"],
}).rename_axis("Año")

,Norma,Calculado,Informe,Diferencia
Año,,,,
2019,NADF-009-AIRE-2017,94,94,0
2020,NOM-172-SEMARNAT-2019,125,125,0
2021,NOM-172-SEMARNAT-2019,135,135,0
2022,NOM-172-SEMARNAT-2019,126,126,0
2023,NOM-172-SEMARNAT-2019,120,120,0


#### 1.2 Días por categoría

In [3]:
cols = ["Malos", "Muy Mala o peor", "Mala", "Muy Mala", "Ext. Mala"]
pd.concat({"Calculado": calculado.loc[anios, cols], "Informe": INFORME[cols],
           "Diferencia": calculado.loc[anios, cols] - INFORME[cols]}, axis=1).rename_axis("Año")

Calculado                                         Informe  \
         Malos Muy Mala o peor Mala Muy Mala Ext. Mala   Malos   
Año                                                              
2019       271              12  259       11         1     271   
2020       241              55  186       54         1     241   
2021       230              55  175       49         6     230   
2022       239              73  166       61        12     239   
2023       245              65  180       61         4     245   

                                              Diferencia                  \
     Muy Mala o peor  Mala Muy Mala Ext. Mala      Malos Muy Mala o peor   
Año                                                                        
2019              12  <NA>     <NA>      <NA>          0               0   
2020              12   229       12         0          0              43   
2021              55   175       49         6          0               0   
2022              72   167       60        12          0               1   
2023              65   180       61         4          0               0   

                               
      Mala Muy Mala Ext. Mala  
Año                            
2019  <NA>     <NA>      <NA>  
2020   -43       42         1  
2021     0        0         0  
2022    -1        1         0  
2023     0        0         0

#### 1.3 Días Mala o peor por contaminante

In [4]:
cols = ["O3", "PM10", "PM2.5"]
pd.concat({"Calculado": calculado.loc[anios, cols], "Informe": INFORME[cols],
           "Diferencia": calculado.loc[anios, cols] - INFORME[cols]}, axis=1).rename_axis("Año")

Calculado            Informe            Diferencia           
            O3 PM10 PM2.5      O3 PM10 PM2.5         O3 PM10 PM2.5
Año                                                               
2019       204  175    55     204  175    55          0    0     0
2020       229   68    10     208   68    10         21    0     0
2021       204  101    13     204  101    13          0    0     0
2022       222   89    10     222   89    10          0    0     0
2023       228   82    10     228   82    10          0    0     0

#### 2. Serie histórica de días limpios

Cada día se evalúa con la norma oficial vigente en su fecha, que es el criterio de los Informes Anuales del SIMAT: NADF-009-AIRE-2017 en 2019, NOM-172-SEMARNAT-2019 de 2020 al 22 de julio de 2024 y NOM-172-SEMARNAT-2023 desde el 23 de julio de 2024. Por eso 2024 combina las dos NOM.

In [5]:
serie = pd.DataFrame({
    "Norma vigente": pd.Series(norma_anio),
    "Días del año": {anio: len(d) for anio, d in diario.items()},
    "Días limpios": calculado["Limpios"],
    "% limpios": {anio: round(100 * (d["Total"] <= 1).mean(), 1) for anio, d in diario.items()},
})
serie.rename_axis("Año")

,Norma vigente,Días del año,Días limpios,% limpios
Año,,,,
2019,NADF-009-AIRE-2017,365,94,25.8
2020,NOM-172-SEMARNAT-2019,366,125,34.2
2021,NOM-172-SEMARNAT-2019,365,135,37.0
2022,NOM-172-SEMARNAT-2019,365,126,34.5
2023,NOM-172-SEMARNAT-2019,365,120,32.9
2024,NOM-172-SEMARNAT-2019 + NOM-172-SEMARNAT-2023,366,88,24.0
2025,NOM-172-SEMARNAT-2023,365,70,19.2


### Días limpios por mes

In [6]:
pd.DataFrame({anio: (d["Total"] <= 1).groupby(d.index.month).sum() for anio, d in diario.items()}).rename_axis("Mes")

,2019,2020,2021,2022,2023,2024,2025
Mes,,,,,,,
1,2,20,14,10,5,12,6
2,2,10,12,11,5,8,3
3,3,4,4,7,7,2,0
4,1,1,4,1,8,2,0
5,1,2,3,2,5,0,2
6,14,9,21,14,7,10,16
7,16,9,12,13,11,17,8
8,5,13,17,12,15,7,3
9,14,16,17,22,9,11,9


#### 3. Normas y fórmulas de cálculo

#### 3.1 Norma aplicada por periodo

| Periodo | Norma | Índice |
|---|---|---|
| Hasta 2018 | NADF-009-AIRE-2006 (Gaceta Oficial del D.F., 29/11/2006) | IMECA (no incluido en este cálculo) |
| 2019 | NADF-009-AIRE-2017 | Índice de Calidad del Aire, escala 0–500; incluye PM2.5 desde el 1/01/2019 |
| 2020 – 22/07/2024 | NOM-172-SEMARNAT-2019 (DOF 20/11/2019) | Índice AIRE Y SALUD |
| Desde 23/07/2024 | NOM-172-SEMARNAT-2023 (DOF 25/01/2024; en vigor 180 días después) | Índice AIRE Y SALUD, con límites más estrictos |

Los criterios de suficiencia y de redondeo también están en la NOM-020-SSA1-2014 (ozono) y la NOM-025-SSA1-2014 (partículas).

#### 3.2 Datos de entrada

- Concentración promedio horaria $C_{e,h}$ de cada estación $e$ y hora $h$, publicada por el SIMAT. La hora 1 corresponde a los minutos de 00:00 a 00:59, y la hora 24 aparece como las 00:00 del día siguiente.
- Unidades según el catálogo del SIMAT: O3, NO2 y SO2 en ppb (se dividen entre 1000 para pasar a ppm), CO en ppm y partículas en µg/m³.
- Se excluyen las estaciones de transporte: ACO (Acolman), INN (Investigaciones Nucleares), MPA (Milpa Alta), AJU (Ajusco) y MON (Montecillo).

#### 3.3 Fórmulas

**Promedio móvil de $n$ horas** (8 h para O3 y CO; 24 h para SO2 y, en 2019, para partículas), con al menos 75% de horas válidas (NOM-172, 5.2.5.2 y 5.2.5.4; NOM-020-SSA1-2014, 5.2.2.2):

$$\bar C^{(n)}_{e,h} = \frac{1}{m}\sum_{i=0}^{n-1} C_{e,h-i}, \qquad m = \text{horas válidas en la ventana} \ge 0.75\,n$$

**Promedio de 24 horas del día $d$** (partículas desde 2020), de la hora 1 a la 24, "a partir de las 00:00 horas" (NOM-025-SSA1-2014, 5.2.2.1.1 y 5.3.1.1; NOM-172-2023, 5.2.5.4):

$$\bar C^{(24)}_{e,d} = \frac{1}{m}\sum_{h=1}^{24} C_{e,d,h}, \qquad m \ge 18$$

**Redondeo** a los decimales de la NOM-172, tabla 2 ($k$ = 3 para O3, NO2 y SO2 en ppm; 2 para CO; 0 para partículas). Si la cifra siguiente es 5 o más, se sube (5.2.4):

$$\operatorname{red}(x,k) = \frac{\lfloor x\cdot 10^{k} + 0.5 \rfloor}{10^{k}}$$

**Categoría** de un valor según los límites superiores $L_1 < L_2 < L_3 < L_4$ de Buena, Aceptable, Mala y Muy Mala:

$$\operatorname{cat}(x) = \#\{\,j : x > L_j\,\} \in \{0{:}\ \text{Buena},\ 1{:}\ \text{Aceptable},\ 2{:}\ \text{Mala},\ 3{:}\ \text{Muy Mala},\ 4{:}\ \text{Ext. Mala}\}$$

**Categoría del día**: la peor entre estaciones $e$, contaminantes e indicadores $p$ y horas $h$ del día (NOM-172, 5.1.2.5 y 5.4.2). Los informes dicen que usan "el valor máximo del día sin considerar la estación, contaminante o indicador":

$$\text{Cat}_d = \max_{e,\,p,\,h \in d} \operatorname{cat}\big(\operatorname{red}(\text{indicador}_{p,e,h})\big)$$

**Día limpio**: $\text{Cat}_d \le 1$, es decir, Buena o Aceptable. En 2019, índice de hasta 100 puntos (Buena o Regular).

#### 3.4 Indicadores y límites por norma

**NADF-009-AIRE-2017 (2019)**: solo O3, PM10 y PM2.5. Se usa el O3 horario y el máximo del promedio móvil de 24 h de partículas. En la tabla, la concentración equivalente a cada nivel del índice:

| Puntos del índice | O3 1 h (ppb) | PM10 móvil 24 h (µg/m³) | PM2.5 móvil 24 h (µg/m³) |
|---|---|---|---|
| 100 (fin de Regular) | 95 | 75 | 45 |
| 150 (fin de Mala) | 154 | 214 | 97.4 |
| 200 (fin de Muy Mala) | 204 | 354 | 150.4 |

**NOM-172-SEMARNAT-2019 (2020 – 22/07/2024)**, límites superiores:

| Categoría | PM10 24 h | PM2.5 24 h | O3 1 h | O3 8 h | NO2 1 h | SO2 24 h | CO 8 h |
|---|---|---|---|---|---|---|---|
| Buena | 50 | 25 | 0.051 | 0.051 | 0.107 | 0.008 | 8.75 |
| Aceptable | 75 | 45 | 0.095 | 0.070 | 0.210 | 0.110 | 11.00 |
| Mala | 155 | 79 | 0.135 | 0.092 | 0.230 | 0.165 | 13.30 |
| Muy Mala | 235 | 147 | 0.175 | 0.114 | 0.250 | 0.220 | 15.50 |

Ozono combinado: la categoría más alta entre el promedio de 1 h y el de 8 h. Para partículas, el informe 2021 (pág. 106) indica que "para el conteo de días se utiliza el promedio de 24 horas".

**NOM-172-SEMARNAT-2023 (desde el 23/07/2024)**, reporte diario (tabla 3): máximo horario de O3, NO2 y SO2; máximo del promedio móvil de 8 h de CO; promedio de 24 h de partículas. El ozono ya no es combinado. Límites superiores:

| Categoría | PM10 24 h (2024–2025 / desde 2026) | PM2.5 24 h (2024–2025 / desde 2026) | O3 1 h | NO2 1 h | SO2 1 h | CO 8 h |
|---|---|---|---|---|---|---|
| Buena | 45 / 45 | 15 / 15 | 0.058 | 0.053 | 0.035 | 5.00 |
| Aceptable | 60 / 50 | 33 / 25 | 0.090 | 0.106 | 0.075 | 9.00 |
| Mala | 132 / 132 | 79 / 79 | 0.135 | 0.160 | 0.185 | 12.00 |
| Muy Mala | 213 / 213 | 130 / 130 | 0.175 | 0.213 | 0.304 | 16.00 |

La norma de 2023 escribe los intervalos como "<45" y ">45 a 70", sin decir en qué categoría cae el valor exacto del límite. Aquí se asigna a la categoría inferior, igual que en la norma de 2019. Para los límites de partículas se usa la columna "a partir de enero de 2024", porque la norma entró en vigor después de esa fecha.

#### 3.5 Particularidades de los archivos

- **2021:** fechas en formato dd/mm/aaaa, con la hora 24 escrita como 24:00.
- **2022:** la hora 24 viene sin fecha; se reconstruye como la hora anterior más una hora. Falta la hora 24 del 31 de diciembre.
- **2022 y 2023:** se publican como ZIP con extensión `.gz`; se renombraron a `.zip`.
- **2025:** la columna de valores se llama `valor` y la hora se marca al inicio (00:00 = hora 1); se desplaza una hora.

#### 4. Prospectiva

#### 4.1 Tasa de variación media anual (TVMA) de días limpios, 2020–2023

Se usa la serie con la norma vigente. De 2020 a 2023 todos los años se calcularon con la NOM-172-SEMARNAT-2019, así que el periodo es homogéneo y coincide con los informes publicados.

$$\text{TVMA} = \left(\frac{V_{2023}}{V_{2020}}\right)^{1/n} - 1, \qquad n = 2023 - 2020 = 3$$

Se calcula sobre los días limpios y sobre el porcentaje de días limpios, porque 2020 fue bisiesto (366 días).

In [7]:
INICIO, FIN = 2020, 2023
periodo = pd.DataFrame({
    "Norma": pd.Series(norma_anio),
    "Días limpios": calculado["Limpios"].astype(float),
    "% limpios": {anio: 100 * (d["Total"] <= 1).mean() for anio, d in diario.items()},
}).loc[INICIO:FIN]
periodo["Variación anual (%)"] = 100 * periodo["Días limpios"].pct_change()


def tvma(serie):
    return 100 * ((serie.iloc[-1] / serie.iloc[0]) ** (1 / (len(serie) - 1)) - 1)


print(f"TVMA {INICIO}-{FIN}, días limpios:   {tvma(periodo['Días limpios']):.2f} % anual")
print(f"TVMA {INICIO}-{FIN}, % días limpios: {tvma(periodo['% limpios']):.2f} % anual")
periodo.round(2).rename_axis("Año")

TVMA 2020-2023, días limpios:   -1.35 % anual
TVMA 2020-2023, % días limpios: -1.26 % anual


,Norma,Días limpios,% limpios,Variación anual (%)
Año,,,,
2020,NOM-172-SEMARNAT-2019,125.0,34.15,NaN
2021,NOM-172-SEMARNAT-2019,135.0,36.99,8.00
2022,NOM-172-SEMARNAT-2019,126.0,34.52,-6.67
2023,NOM-172-SEMARNAT-2019,120.0,32.88,-4.76


### 4.2 Escenarios a 2047

Para la serie de días limpios con la norma vigente se construyen tres escenarios a partir de la **línea base 2025**, el último año con datos completos (2026 aún no tiene datos suficientes y no se usa):

$$\hat V_t = V_{2025}\,(1 + r)^{\,t - 2025}$$

- **Inercial:** $r$ es la TVMA 2020–2023 (sección 4.1), el periodo calculado con una sola norma (NOM-172-SEMARNAT-2019), para que la tasa no incluya el efecto del cambio de norma.
- **Deseable y transformador:** $r$ es la tasa anual de los escenarios de la prospectiva previa de calidad del aire (`datos_medioambiente_20250601.xlsx`, hoja `CalidadAire`, columnas `tendencia_deseable` y `tendencia_disruptivo`). Los escenarios distinguen el nivel de esfuerzo institucional, social y económico dedicado al control de emisiones.

Los valores proyectados se redondean a días enteros con el criterio de la NOM-172-SEMARNAT-2023 (numeral 5.2.4): si el primer decimal es 5 o más, el entero se incrementa (9.4 queda en 9 y 9.5 en 10).

**Ajuste manual de tasas:** el bloque `TVMA_MANUAL_VIGENTE` permite cambiar cada tasa. Con `None` se usa la tasa calculada; con un número se reemplaza por esa tasa anual, escrita como proporción (por ejemplo, `-0.005` es −0.5% anual y `0.012` es +1.2% anual). En la matriz, las tasas modificadas aparecen marcadas como "(manual)".

In [8]:
# Línea base = 2025: último año completo; 2026 aún no tiene datos suficientes
LINEA_BASE, HORIZONTE = 2025, 2047

# Tasas anuales de los escenarios de la prospectiva previa (constantes en todo el horizonte)
EXCEL_ESCENARIOS = Path("datos/datos_medioambiente_20250601.xlsx")
escenarios_previos = pd.read_excel(EXCEL_ESCENARIOS, sheet_name="CalidadAire").set_index("periodo")
TVMA_DESEABLE = escenarios_previos.loc[2025, "tendencia_deseable"] / escenarios_previos.loc[2024, "tendencia_deseable"] - 1
TVMA_TRANSFORMADOR = escenarios_previos.loc[2025, "tendencia_disruptivo"] / escenarios_previos.loc[2024, "tendencia_disruptivo"] - 1


def prospectiva(serie, titulo, inicio_tvma, fin_tvma, tvma_manual=None):
    """Matriz de escenarios: histórico y proyección desde LINEA_BASE hasta HORIZONTE.
    La TVMA inercial se calcula entre inicio_tvma y fin_tvma; tvma_manual reemplaza tasas por escenario."""
    historico = serie.loc[:LINEA_BASE].astype(float).to_dict()
    valor_base = historico[LINEA_BASE]
    tvma_inercial = (historico[fin_tvma] / historico[inicio_tvma]) ** (1 / (fin_tvma - inicio_tvma)) - 1
    tasas = {"inercial": tvma_inercial, "deseable": TVMA_DESEABLE, "transformador": TVMA_TRANSFORMADOR}
    manual = {esc: r for esc, r in (tvma_manual or {}).items() if r is not None}
    tasas.update(manual)
    etiqueta = {esc: f"{r:+.2%}" + (" (manual)" if esc in manual else "") for esc, r in tasas.items()}

    anios = range(min(historico), HORIZONTE + 1)
    df = pd.DataFrame({
        esc: [historico.get(a, np.nan) if a <= LINEA_BASE else valor_base * (1 + r) ** (a - LINEA_BASE) for a in anios]
        for esc, r in tasas.items()
    }, index=pd.Index(anios, name="periodo"))
    df = redondear(df, 0)  # días enteros, NOM-172-SEMARNAT-2023 (5.2.4)

    print("\n" + "=" * 80)
    print(f" MATRIZ DE ESCENARIOS PROSPECTIVOS: {titulo} ({min(historico)}-{HORIZONTE})")
    print(" (Días limpios al año: categoría Buena o Aceptable en todo el día)")
    print(f" Línea base: {LINEA_BASE} = {valor_base:.0f} días | TVMA Observada {inicio_tvma}-{fin_tvma}: {tvma_inercial:+.2%}")
    print(f" TVMA Inercial: {etiqueta['inercial']} | Deseable: {etiqueta['deseable']} | Transformador: {etiqueta['transformador']}")
    print("=" * 80)
    display(df.style.format("{:.0f}", na_rep="-"))
    return df

In [9]:
# Ajuste manual de TVMA, norma vigente: None = tasa calculada; número = tasa anual (-0.005 = -0.5 %)
TVMA_MANUAL_VIGENTE = {
    "inercial": 0.01,
    "deseable": 0.02,
    "transformador": 0.025,
}

prospectiva_vigente = prospectiva(calculado["Limpios"], "DÍAS LIMPIOS, NORMA VIGENTE", 2020, 2023, TVMA_MANUAL_VIGENTE)


 MATRIZ DE ESCENARIOS PROSPECTIVOS: DÍAS LIMPIOS, NORMA VIGENTE (2019-2047)
 (Días limpios al año: categoría Buena o Aceptable en todo el día)
 Línea base: 2025 = 70 días | TVMA Observada 2020-2023: -1.35%
 TVMA Inercial: +1.00% (manual) | Deseable: +2.00% (manual) | Transformador: +2.50% (manual)


,inercial,deseable,transformador
periodo,,,
2019,94,94,94
2020,125,125,125
2021,135,135,135
2022,126,126,126
2023,120,120,120
2024,88,88,88
2025,70,70,70
2026,71,71,72
2027,71,73,74


### 4.3 Prospectiva con la NOM-172-SEMARNAT-2023 en toda la serie

Aquí todos los años (2019–2025) se recalculan con la NOM-172-SEMARNAT-2023, sin importar la norma vigente en cada fecha. La serie queda homogénea y la TVMA no incluye el efecto de los cambios de norma. Los límites de partículas son los de "a partir de enero de 2024", porque la serie termina en 2025.

La TVMA inercial usa el mismo periodo que la sección 4.1 (2020–2023), y la línea base sigue siendo 2025, para que las dos prospectivas sean comparables. Los escenarios deseable y transformador usan las mismas tasas de la prospectiva previa. El bloque `TVMA_MANUAL_2023` funciona igual que `TVMA_MANUAL_VIGENTE`.

In [10]:
diario_2023 = {anio: indice_diario(w, anio, "NOM-172-SEMARNAT-2023") for anio, w in datos.items()}
calculado_2023 = resumen(diario_2023)

serie_2023 = pd.DataFrame({
    "Días limpios, norma vigente": calculado["Limpios"],
    "Días limpios, NOM-2023": calculado_2023["Limpios"],
    "% limpios, NOM-2023": {anio: round(100 * (d["Total"] <= 1).mean(), 1) for anio, d in diario_2023.items()},
})
serie_2023["Variación anual NOM-2023 (%)"] = (100 * serie_2023["Días limpios, NOM-2023"].astype(float).pct_change()).round(2)

print(f"TVMA {INICIO}-{FIN}, días limpios NOM-2023: {tvma(calculado_2023['Limpios'].loc[INICIO:FIN].astype(float)):.2f} % anual")
serie_2023.rename_axis("Año")

TVMA 2020-2023, días limpios NOM-2023: 3.04 % anual


,"Días limpios, norma vigente","Días limpios, NOM-2023","% limpios, NOM-2023",Variación anual NOM-2023 (%)
Año,,,,
2019,94,63,17.3,NaN
2020,125,85,23.2,34.92
2021,135,101,27.7,18.82
2022,126,93,25.5,-7.92
2023,120,93,25.5,0.00
2024,88,76,20.8,-18.28
2025,70,70,19.2,-7.89


In [11]:
# Ajuste manual de TVMA, NOM-172-SEMARNAT-2023: None = tasa calculada; número = tasa anual (-0.005 = -0.5 %)
TVMA_MANUAL_2023 = {
    "inercial": 0.006,
    "deseable": 0.015,
    "transformador": 0.02,
}

prospectiva_2023 = prospectiva(calculado_2023["Limpios"], "DÍAS LIMPIOS, NOM-172-SEMARNAT-2023", INICIO, FIN, TVMA_MANUAL_2023)


 MATRIZ DE ESCENARIOS PROSPECTIVOS: DÍAS LIMPIOS, NOM-172-SEMARNAT-2023 (2019-2047)
 (Días limpios al año: categoría Buena o Aceptable en todo el día)
 Línea base: 2025 = 70 días | TVMA Observada 2020-2023: +3.04%
 TVMA Inercial: +0.60% (manual) | Deseable: +1.50% (manual) | Transformador: +2.00% (manual)


,inercial,deseable,transformador
periodo,,,
2019,63,63,63
2020,85,85,85
2021,101,101,101
2022,93,93,93
2023,93,93,93
2024,76,76,76
2025,70,70,70
2026,70,71,71
2027,71,72,73


In [12]:
Path("resultados").mkdir(exist_ok=True)
for anio, d in diario.items():
    salida = d.drop(columns="Norma").apply(lambda s: s.map(CATEGORIAS)).assign(Norma=d["Norma"])
    if anio == 2019:  # en la NADF, 1 = Buena o Regular (hasta 100 puntos)
        salida = d.drop(columns="Norma").apply(lambda s: s.map({1: "Buena o Regular", 2: "Mala", 3: "Muy Mala", 4: "Ext. Mala o peor"})).assign(Norma=d["Norma"])
    salida.rename_axis("Fecha").to_csv(f"resultados/calculo_simat_{anio}_dias.csv")

prospectiva_vigente.astype("Int64").to_csv("resultados/calculo_simat_escenarios_vigente_2047.csv")
prospectiva_2023.astype("Int64").to_csv("resultados/calculo_simat_escenarios_nom2023_2047.csv")